<a href="https://colab.research.google.com/github/irispilo/INTRO-DATOS/blob/main/CE1_Barista_Lovers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Caso de Estudio 1 — CS070: Introducción a los Datos
## Sistema de Puntos **Barista Lovers** — Café Barista Guatemala

---

**Curso:** CS070 — Introducción a los Datos  
**Empresa analizada:** Café Barista Guatemala (Cantata S.A.)  
**Programa de lealtad:** Barista Lovers (lanzado agosto 2025, reemplaza a "Soy Barista")  
**Rol del equipo:** Consultores de datos contratados para diagnosticar y optimizar el sistema de recompensas  

---

## 1. Resumen de la Situación

### 1.1 Contexto de la Empresa

Café Barista es una cadena de cafeterías premium fundada en Guatemala en 2004, operada por **Cantata S.A.** (subsidiaria de BIA QSR). Cuenta con más de 50 sucursales en Guatemala y ha comenzado expansión regional (El Salvador).  

### 1.2 El Programa Barista Lovers

En agosto de 2025, Café Barista migró de su sistema físico de tarjetas **"Soy Barista"** a un programa digital llamado **"Barista Lovers"**, operado sobre la plataforma **Piggy** (`cafe-barista.app.piggy.eu`). El programa utiliza wallet digital (Apple Wallet / Google Wallet), códigos QR y un portal web.

### 1.3 Reglas de Negocio del Sistema (fuente: cafebarista.com.gt)

| Regla | Detalle |
|---|---|
| **Acumulación** | Q1.00 de compra = 1 punto |
| **Tope de puntos** | 4,000 puntos máximo en saldo activo |
| **Canje mínimo** | Recompensas desde 200 hasta 350 puntos |
| **Niveles** | Silver (inicio), Gold (>1,500 pts históricos), Platinum (>3,500 pts históricos) |
| **Nivel no baja al canjear** | El nivel se basa en historial de compras, no en saldo actual |
| **Caducidad por inactividad** | 100% de puntos se pierden al no tener actividad por 6 meses (corte: 1 junio 2026) |
| **Ajuste anual** | 25% de reducción de saldo al 31 diciembre 2026 para cuentas creadas antes del 1 julio 2026 |
| **Canje** | Solo en tiendas físicas con código QR |
| **Transferencia** | Puntos intransferibles entre cuentas |

### 1.4 Niveles y Beneficios

| Nivel | Umbral (histórico) | Beneficios Adicionales |
|---|---|---|
| 🥈 Pergamino (Silver) | 0 pts (inicio) | Café gratis de bienvenida, bebida de cumpleaños, shot adicional |
| 🥇 Blend Barista (Gold) | > 1,500 pts | + Refill café de la casa, agrandado gratis, puntos dobles en temporadas |
| 💎 Reserva Especial (Platinum) | > 3,500 pts | + Refill Americano, cambio a leche vegetal, reservación de sala, regalos especiales, 10% off retail |

### 1.5 Arquitectura General del Sistema

```
┌─────────────────────────────────────────────────────────────┐
│                     CLIENTE FINAL                           │
│        (App Wallet / Portal Web / Correo electrónico)       │
└──────────────────────┬──────────────────────────────────────┘
                       │  QR Code / Identificación
┌──────────────────────▼──────────────────────────────────────┐
│               PUNTO DE VENTA (Tienda física)                │
│         Escáner QR → POS → Registro de transacción          │
└──────────────────────┬──────────────────────────────────────┘
                       │  API / Webhook
┌──────────────────────▼──────────────────────────────────────┐
│           PLATAFORMA PIGGY (cafe-barista.app.piggy.eu)      │
│  Motor de puntos │ Gestión de niveles │ Emisión de vouchers  │
└──────────────────────┬──────────────────────────────────────┘
                       │
┌──────────────────────▼──────────────────────────────────────┐
│                  CANTATA S.A. (Back-end)                    │
│     CRM │ Análisis de datos │ Comunicaciones (email/push)   │
└─────────────────────────────────────────────────────────────┘
```

**Flujo de datos:** Compra → POS captura monto → API Piggy calcula puntos (Q1=1pt) → actualiza wallet del cliente → evalúa si sube de nivel → emite voucher/recompensa si aplica → notifica por correo.

---
## 2. Configuración del Entorno

In [ ]:
# Instalación de librerías necesarias (solo ejecutar si es necesario)
# !pip install pandas numpy matplotlib seaborn faker --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from datetime import datetime, timedelta
import random
import warnings

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

# Estilo de gráficas
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.family'] = 'DejaVu Sans'
BARISTA_GREEN = '#2E7D32'
BARISTA_BROWN = '#6D4C41'
BARISTA_GOLD  = '#F9A825'
BARISTA_CREAM = '#FFF8E1'

print("Librerías cargadas correctamente")
print(f"Fecha de análisis: {datetime.now().strftime('%d/%m/%Y %H:%M')}")

Librerías cargadas correctamente
Fecha de análisis: 24/04/2026 13:34


---
## 3. Pipeline de Datos — Réplica en Miniatura del Sistema

### 3.1 Extracción (Simulación de Datos)

Dado que no tenemos acceso directo a la base de datos de Piggy/Cantata, generamos datos sintéticos que replican fielmente la estructura real del sistema Barista Lovers.

In [ ]:
# ─────────────────────────────────────────────
# PASO 1: Generación de clientes
# ─────────────────────────────────────────────

N_CLIENTES = 500

nombres = [
    'Ana','Luis','María','Carlos','Sofía','Diego','Fernanda','José',
    'Valeria','Pablo','Isabella','Andrés','Camila','Roberto','Lucía',
    'Miguel','Daniela','Alejandro','Paula','Ricardo'
]
apellidos = [
    'García','López','Martínez','González','Pérez','Rodríguez',
    'Hernández','Díaz','Torres','Morales','Cifuentes','Chávez',
    'Gramajo','Cabrera','Salazar','Méndez','Ríos','Estrada'
]

# Fecha de inscripción al programa (desde agosto 2025)
inicio_programa = datetime(2025, 8, 1)
fecha_analisis  = datetime(2026, 4, 24)  # Fecha actual

clientes = pd.DataFrame({
    'cliente_id': [f'BL{str(i).zfill(5)}' for i in range(1, N_CLIENTES+1)],
    'nombre': [f"{random.choice(nombres)} {random.choice(apellidos)}" for _ in range(N_CLIENTES)],
    'email': [f"usuario{i}@email.com" for i in range(1, N_CLIENTES+1)],
    'fecha_nacimiento': [
        datetime(random.randint(1970, 2004), random.randint(1,12), random.randint(1,28))
        for _ in range(N_CLIENTES)
    ],
    'fecha_registro': [
        inicio_programa + timedelta(days=random.randint(0, (fecha_analisis - inicio_programa).days))
        for _ in range(N_CLIENTES)
    ],
    'activo': np.random.choice([True, False], N_CLIENTES, p=[0.78, 0.22])
})

print(f"Clientes generados: {len(clientes)}")
print(f"   Activos: {clientes['activo'].sum()} | Inactivos: {(~clientes['activo']).sum()}")
clientes.head()

Clientes generados: 500
   Activos: 387 | Inactivos: 113


,cliente_id,nombre,email,fecha_nacimiento,fecha_registro,activo
0,BL00001,Carlos García,usuario1@email.com,1985-11-22,2025-08-31,True
1,BL00002,Valeria Díaz,usuario2@email.com,1988-04-24,2025-09-05,False
2,BL00003,José Pérez,usuario3@email.com,1975-07-04,2026-03-30,True
3,BL00004,Carlos Estrada,usuario4@email.com,1976-08-06,2025-08-17,True
4,BL00005,María Cabrera,usuario5@email.com,1989-01-02,2025-12-25,True


In [ ]:
# ─────────────────────────────────────────────
# PASO 2: Generación de transacciones
# ─────────────────────────────────────────────

productos = {
    'Café Americano 12oz':    35,
    'Café Americano 16oz':    40,
    'Latte 12oz':             45,
    'Latte 16oz':             52,
    'Cappuccino 12oz':        45,
    'Frappé':                 60,
    'Cold Brew':              58,
    'Iced Latte':             55,
    'Té':                     32,
    'Panini':                 65,
    'Croissant':              28,
    'Pound Cake':             30,
    'Combo Desayuno':         85,
    'Bebida Temporada':       65,
    'Refresher':              52,
}

sucursales = [
    'Zona 10', 'Zona 15', 'Oakland Mall', 'Miraflores',
    'Pradera Concepción', 'Gran Vía', 'Majadas Once',
    'El Paseo', 'Cuatro Grados Norte', 'Cayalá'
]

transacciones = []

for _, cliente in clientes.iterrows():
    # Clientes activos compran más
    n_compras = (
        random.randint(5, 80) if cliente['activo']
        else random.randint(0, 4)
    )
    for _ in range(n_compras):
        fecha_tx = cliente['fecha_registro'] + timedelta(
            days=random.randint(0, (fecha_analisis - cliente['fecha_registro']).days)
        )
        producto  = random.choice(list(productos.keys()))
        monto     = productos[producto] * random.randint(1, 3)  # 1 a 3 unidades
        puntos    = monto  # Q1 = 1 punto

        transacciones.append({
            'tx_id':       f'TX{len(transacciones)+1:07d}',
            'cliente_id':  cliente['cliente_id'],
            'fecha':       fecha_tx,
            'producto':    producto,
            'monto_gtq':   monto,
            'puntos_gen':  puntos,
            'sucursal':    random.choice(sucursales),
            'tipo':        'acumulacion'
        })

df_tx = pd.DataFrame(transacciones)
df_tx['fecha'] = pd.to_datetime(df_tx['fecha'])

print(f" Transacciones generadas: {len(df_tx):,}")
print(f" Monto total facturado: Q{df_tx['monto_gtq'].sum():,.2f}")
print(f" Puntos totales generados: {df_tx['puntos_gen'].sum():,}")
df_tx.head()

 Transacciones generadas: 17,012
 Monto total facturado: Q1,705,770.00
 Puntos totales generados: 1,705,770


,tx_id,cliente_id,fecha,producto,monto_gtq,puntos_gen,sucursal,tipo
0,TX0000001,BL00001,2025-11-12,Latte 12oz,45,45,Gran Vía,acumulacion
1,TX0000002,BL00001,2026-04-03,Iced Latte,165,165,Zona 10,acumulacion
2,TX0000003,BL00001,2026-04-09,Frappé,60,60,El Paseo,acumulacion
3,TX0000004,BL00001,2025-10-04,Latte 16oz,104,104,Cuatro Grados Norte,acumulacion
4,TX0000005,BL00001,2026-01-07,Croissant,56,56,Majadas Once,acumulacion


### 3.2 Limpieza y Validación de Datos

In [ ]:
# ─────────────────────────────────────────────
# PASO 3: Limpieza de datos
# ─────────────────────────────────────────────

print("=" * 50)
print("REPORTE DE CALIDAD — TRANSACCIONES")
print("=" * 50)
print(f"Registros totales     : {len(df_tx):,}")
print(f"Valores nulos         : {df_tx.isnull().sum().sum()}")
print(f"Duplicados            : {df_tx.duplicated('tx_id').sum()}")
print(f"Monto mínimo          : Q{df_tx['monto_gtq'].min()}")
print(f"Monto máximo          : Q{df_tx['monto_gtq'].max()}")
print(f"Rango de fechas       : {df_tx['fecha'].min().date()} → {df_tx['fecha'].max().date()}")

# Validar que no hay montos negativos o cero
invalidos = df_tx[df_tx['monto_gtq'] <= 0]
print(f"Transacciones inválidas (monto ≤ 0): {len(invalidos)}")

# Limpiar: remover duplicados e inválidos
df_tx_clean = df_tx.drop_duplicates('tx_id')
df_tx_clean = df_tx_clean[df_tx_clean['monto_gtq'] > 0]

print(f"\n Registros limpios : {len(df_tx_clean):,}")

REPORTE DE CALIDAD — TRANSACCIONES
Registros totales     : 17,012
Valores nulos         : 0
Duplicados            : 0
Monto mínimo          : Q28
Monto máximo          : Q255
Rango de fechas       : 2025-08-02 → 2026-04-24
Transacciones inválidas (monto ≤ 0): 0

 Registros limpios : 17,012


### 3.3 Transformación y Cálculo de Puntos

In [ ]:
# ─────────────────────────────────────────────
# PASO 4: Transformación — Motor de Puntos
# ─────────────────────────────────────────────

TOPE_PUNTOS         = 4000   # Saldo máximo permitido
UMBRAL_GOLD         = 1500   # Puntos históricos para Gold
UMBRAL_PLATINUM     = 3500   # Puntos históricos para Platinum

def calcular_nivel(puntos_historicos: int) -> str:
    """Determina el nivel según historial acumulado (no saldo actual)."""
    if puntos_historicos >= UMBRAL_PLATINUM:
        return ' Reserva Especial (Platinum)'
    elif puntos_historicos >= UMBRAL_GOLD:
        return ' Blend Barista (Gold)'
    else:
        return ' Pergamino (Silver)'

def aplicar_tope(saldo: int) -> int:
    """Respeta el tope de 4,000 puntos en saldo activo."""
    return min(saldo, TOPE_PUNTOS)

# Resumen por cliente
resumen = df_tx_clean.groupby('cliente_id').agg(
    total_compras    = ('tx_id',        'count'),
    gasto_total_gtq  = ('monto_gtq',    'sum'),
    puntos_historicos= ('puntos_gen',   'sum'),
    ultima_compra    = ('fecha',        'max'),
    primera_compra   = ('fecha',        'min'),
).reset_index()

# Saldo activo (con tope)
resumen['saldo_activo'] = resumen['puntos_historicos'].apply(aplicar_tope)

# Nivel
resumen['nivel'] = resumen['puntos_historicos'].apply(calcular_nivel)

# Días desde última compra
resumen['dias_inactividad'] = (fecha_analisis - resumen['ultima_compra']).dt.days

# Clientes en riesgo de perder puntos (>= 150 días inactivos → riesgo alto antes del corte junio)
resumen['riesgo_caducidad'] = resumen['dias_inactividad'] >= 150

# Merge con info de cliente
df_final = resumen.merge(clientes[['cliente_id','nombre','activo']], on='cliente_id')

print(f" Tabla resumen construida: {len(df_final)} clientes")
df_final.head(10)

 Tabla resumen construida: 468 clientes


,cliente_id,total_compras,gasto_total_gtq,puntos_historicos,ultima_compra,primera_compra,saldo_activo,nivel,dias_inactividad,riesgo_caducidad,nombre,activo
0,BL00001,67,6385,6385,2026-04-23,2025-08-31,4000,Reserva Especial (Platinum),1,False,Carlos García,True
1,BL00002,1,120,120,2026-03-16,2026-03-16,120,Pergamino (Silver),39,False,Valeria Díaz,False
2,BL00003,49,4746,4746,2026-04-24,2026-03-30,4000,Reserva Especial (Platinum),0,False,José Pérez,True
3,BL00004,44,3680,3680,2026-04-23,2025-08-18,3680,Reserva Especial (Platinum),1,False,Carlos Estrada,True
4,BL00005,55,5589,5589,2026-04-23,2025-12-25,4000,Reserva Especial (Platinum),1,False,María Cabrera,True
5,BL00006,53,4965,4965,2026-04-24,2026-02-28,4000,Reserva Especial (Platinum),0,False,Luis García,True
6,BL00007,56,5590,5590,2026-04-21,2025-11-06,4000,Reserva Especial (Platinum),3,False,María Hernández,True
7,BL00008,1,120,120,2026-02-14,2026-02-14,120,Pergamino (Silver),69,False,José Ríos,False
8,BL00009,72,7022,7022,2026-04-24,2026-03-04,4000,Reserva Especial (Platinum),0,False,Ricardo García,True
9,BL00010,71,8456,8456,2026-04-24,2026-02-08,4000,Reserva Especial (Platinum),0,False,Alejandro Hernández,True


---
## 4. Código / POC — Prueba Funcional del Sistema de Puntos

### 4.1 Simulación del Motor de Puntos Completo

In [ ]:
# ─────────────────────────────────────────────────────────────────────
#  POC: Motor de Puntos Barista Lovers
#  Replica las reglas de negocio reales del programa
# ─────────────────────────────────────────────────────────────────────

class BaristaLoversMotor:
    """
    Réplica del motor de puntos Barista Lovers (Cantata S.A.).
    Reglas según Términos y Condiciones publicados en cafebarista.com.gt
    """

    TOPE_SALDO         = 4_000   # Saldo máximo activo
    UMBRAL_GOLD        = 1_500   # Histórico para nivel Gold
    UMBRAL_PLATINUM    = 3_500   # Histórico para nivel Platinum
    RECOMPENSA_MIN     = 200     # Puntos mínimos para canje
    RECOMPENSA_MAX     = 350     # Puntos máximos para canje básico

    BENEFICIOS = {
        'Silver':   ['Café gratis de bienvenida', 'Bebida de cumpleaños', 'Shot adicional'],
        'Gold':     ['Refill café de la casa', 'Agrandado gratis', 'Puntos dobles (temporada)'],
        'Platinum': ['Refill Americano', 'Leche vegetal gratis', 'Reservación de sala',
                     'Regalos especiales', '10% off en retail'],
    }

    def __init__(self, cliente_id: str, nombre: str):
        self.cliente_id        = cliente_id
        self.nombre            = nombre
        self.saldo_activo      = 0
        self.puntos_historicos = 0
        self.transacciones     = []
        self.nivel             = 'Silver'

    def registrar_compra(self, monto_gtq: float, sucursal: str, fecha: datetime = None) -> dict:
        """Procesa una compra y acredita puntos. Q1 = 1 punto."""
        if monto_gtq <= 0:
            raise ValueError("El monto de compra debe ser mayor a Q0.")

        puntos_nuevos      = int(monto_gtq)  # Q1 = 1pt, ignorar centavos
        saldo_antes        = self.saldo_activo
        historico_antes    = self.puntos_historicos

        # Actualizar puntos históricos (sin tope — determina el nivel)
        self.puntos_historicos += puntos_nuevos

        # Actualizar saldo activo (con tope de 4,000)
        self.saldo_activo = min(self.saldo_activo + puntos_nuevos, self.TOPE_SALDO)
        puntos_acreditados = self.saldo_activo - saldo_antes

        # Actualizar nivel
        nivel_anterior = self.nivel
        self.nivel = self._calcular_nivel()
        subio_nivel = self.nivel != nivel_anterior

        tx = {
            'fecha':               fecha or datetime.now(),
            'tipo':                'compra',
            'monto_gtq':           monto_gtq,
            'puntos_acreditados':  puntos_acreditados,
            'puntos_bloqueados':   puntos_nuevos - puntos_acreditados,
            'saldo_resultante':    self.saldo_activo,
            'sucursal':            sucursal,
            'nivel_resultante':    self.nivel,
        }
        self.transacciones.append(tx)

        resultado = {'status': 'OK', **tx, 'subio_nivel': subio_nivel}
        if subio_nivel:
            resultado['mensaje'] = f'¡Felicidades {self.nombre}! Subiste a nivel {self.nivel}'
        return resultado

    def canjear_recompensa(self, costo_puntos: int, descripcion: str) -> dict:
        """Canjea una recompensa. El nivel NO baja al canjear."""
        if costo_puntos < self.RECOMPENSA_MIN:
            return {'status': 'ERROR', 'mensaje': f'Mínimo de canje: {self.RECOMPENSA_MIN} puntos.'}
        if costo_puntos > self.saldo_activo:
            return {'status': 'ERROR', 'mensaje': f'Saldo insuficiente. Tienes {self.saldo_activo} pts.'}

        self.saldo_activo -= costo_puntos
        # IMPORTANTE: puntos_historicos NO cambia — el nivel se mantiene
        tx = {
            'fecha':             datetime.now(),
            'tipo':              'canje',
            'recompensa':        descripcion,
            'puntos_canjeados':  costo_puntos,
            'saldo_resultante':  self.saldo_activo,
            'nivel_mantenido':   self.nivel,  # No cambia
        }
        self.transacciones.append(tx)
        return {'status': 'OK', 'mensaje': f' Recompensa "{descripcion}" canjeada.', **tx}

    def _calcular_nivel(self) -> str:
        if self.puntos_historicos >= self.UMBRAL_PLATINUM:
            return 'Platinum'
        elif self.puntos_historicos >= self.UMBRAL_GOLD:
            return 'Gold'
        return 'Silver'

    def estado(self) -> dict:
        """Retorna el estado actual de la cuenta."""
        beneficios_actuales = self.BENEFICIOS.get(self.nivel, [])
        # En Gold y Platinum incluyen beneficios de niveles anteriores
        if self.nivel in ('Gold', 'Platinum'):
            beneficios_actuales = self.BENEFICIOS['Silver'] + beneficios_actuales
        if self.nivel == 'Platinum':
            beneficios_actuales = self.BENEFICIOS['Silver'] + self.BENEFICIOS['Gold'] + self.BENEFICIOS['Platinum']

        return {
            'cliente_id':         self.cliente_id,
            'nombre':             self.nombre,
            'nivel':              self.nivel,
            'saldo_activo':       self.saldo_activo,
            'puntos_historicos':  self.puntos_historicos,
            'num_transacciones':  len(self.transacciones),
            'beneficios':         beneficios_actuales,
        }

print(" Clase BaristaLoversMotor definida")

In [ ]:
# ─────────────────────────────────────────────
# Escenario de prueba — cliente completo
# ─────────────────────────────────────────────

print("=" * 60)
print("POC: SIMULACIÓN DE CICLO COMPLETO DE UN CLIENTE")
print("=" * 60)

cliente = BaristaLoversMotor('BL00001', 'Andrea García')

# Compras a lo largo del tiempo
compras = [
    (45,  'Zona 10',           datetime(2025, 8,  5)),   # Latte
    (35,  'Oakland Mall',      datetime(2025, 8, 12)),   # Americano
    (85,  'Cayalá',            datetime(2025, 8, 20)),   # Combo
    (120, 'Miraflores',        datetime(2025, 9,  3)),   # Varias bebidas
    (200, 'Gran Vía',          datetime(2025, 9, 18)),   # Compra grande
    (650, 'Zona 15',           datetime(2025,10,  5)),   # Compra empresarial
    (400, 'Pradera Concepción',datetime(2025,10, 22)),
    (500, 'Oakland Mall',      datetime(2025,11,  8)),
    (350, 'Cayalá',            datetime(2025,11, 25)),
    (420, 'Zona 10',           datetime(2025,12,  3)),
    (180, 'Zona 10',           datetime(2026,  1, 10)),
    (300, 'Gran Vía',          datetime(2026,  2,  5)),
    (250, 'Miraflores',        datetime(2026,  3, 20)),
    (150, 'Zona 10',           datetime(2026,  4, 15)),
]

for monto, sucursal, fecha in compras:
    r = cliente.registrar_compra(monto, sucursal, fecha)
    msg = r.get('mensaje', '')
    print(f"  [{fecha.strftime('%Y-%m-%d')}] Q{monto:>5} en {sucursal:<22} "
          f"→ +{r['puntos_acreditados']:>3} pts | Saldo: {r['saldo_resultante']:>4} pts "
          f"| {r['nivel_resultante']}  {msg}")

print()
# Canje de recompensa
canje = cliente.canjear_recompensa(250, 'Frappé grande gratis')
print(f"  CANJE: {canje['mensaje']} | Saldo tras canje: {canje['saldo_resultante']} pts | Nivel: {canje['nivel_mantenido']}")

print()
# Estado final
estado = cliente.estado()
print("=" * 60)
print("ESTADO FINAL DE CUENTA")
print("=" * 60)
for k, v in estado.items():
    if k != 'beneficios':
        print(f"  {k:<22}: {v}")
print(f"  {'beneficios':<22}: {', '.join(estado['beneficios'][:3])}...")

In [ ]:
# ─────────────────────────────────────────────
# Pruebas de casos borde (edge cases)
# ─────────────────────────────────────────────

print("=" * 60)
print("PRUEBAS DE CASOS BORDE")
print("=" * 60)

# Caso 1: Tope de 4,000 puntos
c_tope = BaristaLoversMotor('BLTST01', 'Test Tope')
c_tope.registrar_compra(3800, 'Zona 10')
r_exceso = c_tope.registrar_compra(1000, 'Zona 10')  # Intento de superar tope
print(f"  Tope 4000 pts | Saldo: {r_exceso['saldo_resultante']} | "
      f"Bloqueados: {r_exceso['puntos_bloqueados']} ")

# Caso 2: Canje insuficiente
c_pobre = BaristaLoversMotor('BLTST02', 'Test Pobre')
c_pobre.registrar_compra(100, 'Zona 10')
r_fallo = c_pobre.canjear_recompensa(250, 'Bebida gratis')
print(f"  Canje con saldo insuficiente: {r_fallo['status']} — {r_fallo['mensaje']} ")

# Caso 3: Nivel no baja al canjear
c_nivel = BaristaLoversMotor('BLTST03', 'Test Nivel')
c_nivel.registrar_compra(2000, 'Zona 10')
nivel_antes = c_nivel.nivel
c_nivel.canjear_recompensa(300, 'Café gratis')
nivel_despues = c_nivel.nivel
print(f"  Nivel antes canje: {nivel_antes} | Nivel después: {nivel_despues} "
      f"({' Sin cambio' if nivel_antes == nivel_despues else 'ERROR'})")

# Caso 4: Monto inválido
c_invalido = BaristaLoversMotor('BLTST04', 'Test Inválido')
try:
    c_invalido.registrar_compra(-50, 'Zona 10')
    print("  Monto negativo: No se capturó el error")
except ValueError as e:
    print(f"  Monto negativo capturado:  '{e}'")

---
## 5. Análisis Exploratorio y Visualizaciones

In [ ]:
# ─────────────────────────────────────────────
# Distribución de niveles
# ─────────────────────────────────────────────

nivel_counts = df_final['nivel'].value_counts()
colores = [BARISTA_GOLD, '#C0C0C0', '#CD7F32']  # Gold, Silver, Bronze

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(BARISTA_CREAM)
for ax in axes:
    ax.set_facecolor(BARISTA_CREAM)

# Pie chart
axes[0].pie(
    nivel_counts.values,
    labels=nivel_counts.index,
    autopct='%1.1f%%',
    colors=[BARISTA_GOLD, '#A5D6A7', '#90CAF9'],
    startangle=140,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
axes[0].set_title('Distribución de Clientes por Nivel', fontsize=13, fontweight='bold', color=BARISTA_BROWN)

# Bar chart: gasto promedio por nivel
gasto_nivel = df_final.groupby('nivel')['gasto_total_gtq'].mean().sort_values(ascending=True)
bars = axes[1].barh(gasto_nivel.index, gasto_nivel.values,
                     color=[BARISTA_GREEN, BARISTA_GOLD, BARISTA_BROWN])
axes[1].set_xlabel('Gasto Promedio (GTQ)', color=BARISTA_BROWN)
axes[1].set_title('Gasto Promedio por Nivel de Cliente', fontsize=13, fontweight='bold', color=BARISTA_BROWN)
for bar, val in zip(bars, gasto_nivel.values):
    axes[1].text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
                  f'Q{val:,.0f}', va='center', fontweight='bold', color=BARISTA_BROWN)

plt.suptitle(' Análisis de Clientes — Barista Lovers', fontsize=15, fontweight='bold',
             color=BARISTA_BROWN, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# Volumen de transacciones por mes y sucursal
# ─────────────────────────────────────────────

df_tx_clean['mes'] = df_tx_clean['fecha'].dt.to_period('M')
ventas_mes = df_tx_clean.groupby('mes').agg(
    monto=('monto_gtq', 'sum'),
    txs=('tx_id', 'count')
).reset_index()
ventas_mes['mes_str'] = ventas_mes['mes'].astype(str)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.patch.set_facecolor(BARISTA_CREAM)
for ax in axes:
    ax.set_facecolor(BARISTA_CREAM)

# Ventas mensuales
axes[0].bar(ventas_mes['mes_str'], ventas_mes['monto'],
             color=BARISTA_GREEN, edgecolor='white')
axes[0].set_title('Facturación Mensual (GTQ)', fontweight='bold', color=BARISTA_BROWN)
axes[0].set_xlabel('Mes')
axes[0].set_ylabel('GTQ')
axes[0].tick_params(axis='x', rotation=45)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'Q{x:,.0f}'))

# Top sucursales
top_suc = df_tx_clean.groupby('sucursal')['monto_gtq'].sum().sort_values(ascending=True)
axes[1].barh(top_suc.index, top_suc.values, color=BARISTA_BROWN)
axes[1].set_title('Facturación por Sucursal', fontweight='bold', color=BARISTA_BROWN)
axes[1].set_xlabel('GTQ total')
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'Q{x:,.0f}'))

plt.suptitle('Rendimiento del Programa por Tiempo y Sucursal', fontsize=14,
             fontweight='bold', color=BARISTA_BROWN)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# Análisis de riesgo: caducidad de puntos
# ─────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(BARISTA_CREAM)
for ax in axes:
    ax.set_facecolor(BARISTA_CREAM)

# Histograma de días de inactividad
axes[0].hist(df_final['dias_inactividad'], bins=30,
              color=BARISTA_BROWN, edgecolor='white', alpha=0.85)
axes[0].axvline(150, color='red', linestyle='--', linewidth=2, label='Umbral riesgo (150 días)')
axes[0].axvline(180, color='darkred', linestyle='-', linewidth=2, label='Caducidad total (180 días)')
axes[0].set_title('Distribución de Días de Inactividad', fontweight='bold', color=BARISTA_BROWN)
axes[0].set_xlabel('Días desde última compra')
axes[0].set_ylabel('Cantidad de clientes')
axes[0].legend()

# Clientes en riesgo por nivel
riesgo_nivel = df_final[df_final['riesgo_caducidad']].groupby('nivel').size()
axes[1].bar(riesgo_nivel.index, riesgo_nivel.values,
             color=[BARISTA_GOLD, '#EF5350', BARISTA_BROWN])
axes[1].set_title('Clientes en Riesgo de Perder Puntos\n(+150 días inactivos)',
                    fontweight='bold', color=BARISTA_BROWN)
axes[1].set_ylabel('Cantidad de clientes')
for i, v in enumerate(riesgo_nivel.values):
    axes[1].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

pts_riesgo = df_final[df_final['riesgo_caducidad']]['saldo_activo'].sum()
plt.suptitle(f'Análisis de Riesgo de Caducidad | Puntos en riesgo: {pts_riesgo:,} pts',
             fontsize=13, fontweight='bold', color='darkred')
plt.tight_layout()
plt.show()

print(f"\n  Clientes en riesgo alto de perder puntos: {df_final['riesgo_caducidad'].sum()}")
print(f"   Puntos totales en riesgo: {pts_riesgo:,} puntos")
print(f"   Equivalente en valor: Q{pts_riesgo:,} en compras no incentivadas")

In [ ]:
# ─────────────────────────────────────────────
# Top productos y ticket promedio
# ─────────────────────────────────────────────

top_prod = df_tx_clean.groupby('producto').agg(
    cantidad = ('tx_id', 'count'),
    ingresos = ('monto_gtq', 'sum')
).sort_values('ingresos', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor(BARISTA_CREAM)
ax.set_facecolor(BARISTA_CREAM)

colores_prod = [BARISTA_GREEN if i < 3 else BARISTA_BROWN for i in range(len(top_prod))]
bars = ax.bar(top_prod.index, top_prod['ingresos'], color=colores_prod, edgecolor='white')
ax.set_title('Top 10 Productos por Ingresos Generados', fontsize=13, fontweight='bold', color=BARISTA_BROWN)
ax.set_ylabel('Ingresos (GTQ)')
ax.tick_params(axis='x', rotation=40)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'Q{x:,.0f}'))

for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
             f'Q{bar.get_height():,.0f}', ha='center', fontsize=8, color=BARISTA_BROWN)

verde_patch = mpatches.Patch(color=BARISTA_GREEN, label='Top 3 productos')
cafe_patch  = mpatches.Patch(color=BARISTA_BROWN, label='Resto del top 10')
ax.legend(handles=[verde_patch, cafe_patch])
plt.tight_layout()
plt.show()

---
## 6. Hallazgos y Diagnóstico del Sistema

### 6.1 Fortalezas del Sistema Actual

| # | Fortaleza |
|---|---|
| 1 | Regla simple de acumulación (Q1 = 1pt) — fácil de entender para el cliente |
| 2 | Niveles basados en historial, no en saldo → el cliente no penalizado por canjear |
| 3 | Plataforma digital (Piggy) — elimina tarjeta física y permite trazabilidad |
| 4 | Múltiples canales de canje (email, portal, wallet) |
| 5 | Beneficios diferenciados y con valor real (leche vegetal, reservación de sala) |

### 6.2 Debilidades y Oportunidades de Optimización

| # | Debilidad | Propuesta de Mejora |
|---|---|---|
| 1 | Tope de 4,000 pts puede frustrar a usuarios frecuentes de alto gasto | Elevar tope o crear sub-categoría de "Super Platinum" |
| 2 | Canje solo en tienda física — excluye a usuarios de delivery (Uber Eats) | Habilitar canje digital para pedidos en app/delivery |
| 3 | Política de caducidad de puntos agresiva (100% a 6 meses inactividad) | Implementar alertas proactivas a los 3 y 5 meses |
| 4 | No hay acumulación en canales digitales (Uber Eats) | Integrar API con plataformas de delivery |
| 5 | Niveles se reinician anualmente → desmotivador para clientes frecuentes | Considerar protección de nivel si se mantiene actividad mínima |
| 6 | No se documenta tasa de conversión nivel Silver → Gold → Platinum | Agregar KPIs de conversión al dashboard |

### 6.3 KPIs Clave del Sistema

In [ ]:
# ─────────────────────────────────────────────
# KPIs del sistema
# ─────────────────────────────────────────────

total_clientes    = len(df_final)
clientes_activos  = df_final['activo'].sum()
tasa_activacion   = clientes_activos / total_clientes * 100
ticket_promedio   = df_tx_clean['monto_gtq'].mean()
compras_promedio  = df_final['total_compras'].mean()
gasto_promedio    = df_final['gasto_total_gtq'].mean()
pts_en_riesgo     = df_final[df_final['riesgo_caducidad']]['saldo_activo'].sum()
pct_platinum      = (df_final['nivel'].str.contains('Platinum').sum() / total_clientes) * 100
pct_gold          = (df_final['nivel'].str.contains('Gold').sum() / total_clientes) * 100
retencion         = clientes_activos / total_clientes * 100

print("=" * 55)
print("     DASHBOARD KPIs — BARISTA LOVERS")
print("=" * 55)
print(f"   Total de clientes registrados : {total_clientes:>6,}")
print(f"   Clientes activos               : {clientes_activos:>6,} ({tasa_activacion:.1f}%)")
print(f"   Ticket promedio por transacción: Q{ticket_promedio:>6.2f}")
print(f"   Compras promedio por cliente   : {compras_promedio:>6.1f} compras")
print(f"   Gasto total promedio / cliente : Q{gasto_promedio:>6.0f}")
print(f"   Clientes Platinum              : {pct_platinum:>6.1f}% del total")
print(f"   Clientes Gold                  : {pct_gold:>6.1f}% del total")
print(f"    Clientes en riesgo caducidad   : {df_final['riesgo_caducidad'].sum():>6,}")
print(f"   Puntos totales en riesgo        : {pts_en_riesgo:>6,} pts")
print(f"   Tasa de retención estimada      : {retencion:>6.1f}%")
print("=" * 55)

---
## 7. Conclusiones

### 7.1 Diagnóstico General

El programa **Barista Lovers** de Café Barista representa una **evolución significativa** respecto a su sistema anterior (tarjeta física "Soy Barista"). La migración a una plataforma digital (Piggy) permite trazabilidad completa de transacciones, comunicación automatizada y gestión dinámica de niveles. Sin embargo, el análisis revela oportunidades críticas de optimización:

1. **Riesgo de churning silencioso:** Los clientes inactivos por más de 6 meses pierden el 100% de sus puntos. Una estrategia de reactivación proactiva (campañas automáticas a los 90 y 150 días de inactividad) podría recuperar entre 15-25% de estos usuarios.

2. **Fricción en el canje digital:** La restricción de canje exclusivo en tienda física desconecta al sistema de los canales de delivery (Uber Eats), que representan una porción creciente del negocio en Guatemala.

3. **Concentración de valor en niveles altos:** Los clientes Platinum generan proporcionalmente mucho más gasto. Optimizar la transición Silver → Gold con incentivos puntuales podría aumentar el gasto total del programa en un 20-30%.

4. **El motor de puntos es escalable:** La regla Q1 = 1 punto es simple y justa, pero podría complementarse con multiplicadores por categoría de producto o por horario ("happy hour de puntos") para distribuir la demanda.

### 7.2 Propuestas de Optimización (Prioridad)

| Prioridad | Acción | Impacto Esperado |
|---|---|---|
| 🔴 Alta | Alertas automáticas de caducidad (90 y 150 días) | Reducir fuga de puntos y churning |
| 🔴 Alta | Integrar canje con canal de delivery | Mayor cobertura del programa |
| 🟡 Media | Multiplicador de puntos por producto premium | Aumentar ticket promedio |
| 🟡 Media | Dashboard de conversión de niveles en tiempo real | Mejor visibilidad del negocio |
| 🟢 Baja | Programa de referidos con puntos | Crecimiento orgánico de la base |

### 7.3 Observación sobre la Arquitectura de Datos

El sistema actual depende de **Piggy** como plataforma de terceros, lo que limita la propiedad de los datos. Se recomienda a Cantata S.A. negociar acceso a datos raw (vía API o exportaciones periódicas) para construir su propio Data Warehouse y no depender exclusivamente del dashboard de Piggy para la toma de decisiones.

---

## 8. Referencias

1. Café Barista Guatemala. (2025). *Barista Lovers — Programa de Lealtad*. https://cafebarista.com.gt/baristalovers/
2. Café Barista Guatemala. (2025). *Términos y Condiciones Programa Barista Lovers*. Cantata S.A. https://cafebarista.com.gt/condiciones/
3. CRISP-DM. (2000). *Cross Industry Standard Process for Data Mining*. IBM / SPSS.
4. McKinsey & Company. (2023). *Loyalty program best practices in the food service industry*.
5. Piggy Loyalty Platform. (2025). https://cafe-barista.app.piggy.eu
6. BIA QSR / Cantata S.A. (2025). https://biaqsr.com/

---

## 9. Repositorio GitHub

> 📁 **Link al repositorio:** `https://github.com/irispilo/INTRO-DATOS`

---

*Notebook desarrollado para el curso CS070 — Introducción a los Datos.*  
*Datos generados sintéticamente con fines académicos. Reglas de negocio basadas en fuentes oficiales de Café Barista Guatemala.*
